<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Quantum_Physics_Higgs_Mechanism_Feynman_Diagrams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Higgs Mechanism — Scientific Animation

### Author: Mugambi Ndwiga
**Instagram:** [@craftsandengineering](https://www.instagram.com/craftsandengineering)

---

## Project Overview
This project generates a polished, self-contained educational MP4 animation explaining the **Higgs Mechanism**, electroweak symmetry breaking, and key particle physics concepts. The animation is built entirely using `matplotlib` and `numpy` in Google Colab.

## Animation Scenes
1.  **The Higgs Field:** Visualizing how particles acquire inertial mass through field interaction.
2.  **Spontaneous Symmetry Breaking:** The 'Mexican Hat' potential and the vacuum settling into a stable state.
3.  **Gluon Fusion:** The dominant Higgs production mode at the LHC via virtual top-quark loops.
4.  **Higgs Decay to Photons (H → ̳̳):** A key discovery channel involving quantum loops.
5.  **The Golden Channel (H → ZZ → 4ℓ):** The remarkably clean signature of four charged leptons.
6.  **Electroweak Symmetry Breaking:** How the Higgs field differentiates the electroweak force into electromagnetism and the weak interaction.

## Technical Specifications
- **Resolution:** 1280×720 (720p)
- **Frame Rate:** 30 FPS
- **Runtime:** ~54 seconds
- **Libraries:** Matplotlib, NumPy
- **Output:** `higgs_mechanism.mp4`

---

## Addendum: Experimental 3D Visualizations
In addition to the standard 2D educational short, this notebook contains high-fidelity 3D visualizations exploring the complex geometry of the Higgs sector:

*   **3D Mexican Hat Potential:** A spatial representation of the Higgs potential $V(\phi)$, demonstrating the transition from a symmetric unstable peak to the degenerate 'circle of minima' that defines the vacuum expectation value (VEV).
*   **3D Gluon Fusion (LHC):** A visualization of non-planar gluon interaction, showing the convergence of field vectors into the virtual top-quark loop to produce the Higgs scalar in a 3D volume.
*   **Kinematic Perspective:** These scenes use rotating camera perspectives to provide better spatial intuition for the symmetry-breaking mechanism.

In [7]:
"""
The Higgs Mechanism — Scientific Animation (Final Verified Version)

Author:
Mugambi Ndwiga

Instagram:
@craftsandengineering

Description:
Educational scientific animation explaining the Higgs field,
electroweak symmetry breaking, Higgs boson production,
and decay channels using particle physics Feynman-style visuals.

Platform:
Google Colab

Libraries:
matplotlib, numpy
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import matplotlib.patches as patches
from matplotlib.patches import Circle
from IPython.display import HTML, display
from base64 import b64encode
import google.colab.files

# ============================================================
# CONFIG
# ============================================================

BG_COLOR = '#050510'

HIGGS_COLOR = '#ffb300'
TOP_COLOR = '#ff7043'
GLUON_COLOR = '#76ff03'
W_COLOR = '#d500f9'
Z_COLOR = '#aa66ff'
PHOTON_COLOR = '#ffd740'
FERMION_COLOR = '#80deea'

TEXT_COLOR = '#f5f5f5'
FLASH_COLOR = '#ffffff'
SUB_TEXT_COLOR = '#b0bec5'

FPS = 30
DURATION = 54
TOTAL_FRAMES = FPS * DURATION

# ============================================================
# HELPERS
# ============================================================

def ease_in_out(t):
    return 0.5 - 0.5 * np.cos(np.pi * np.clip(t, 0, 1))

def lerp(a, b, t):
    return a + (b - a) * t

def fade(t, a, b):
    return np.clip((t - a) / (b - a), 0, 1)

def get_wavy_line(p1, p2, n=200, amp=0.08, freq=10, t_max=1.0):
    p1, p2 = np.array(p1), np.array(p2)
    dist = np.linalg.norm(p2 - p1)
    if dist == 0:
        return np.zeros((n, 2))

    t = np.linspace(0, t_max, n)
    unit = (p2 - p1) / dist
    perp = np.array([-unit[1], unit[0]])

    out = np.zeros((n, 2))

    for i, ti in enumerate(t):
        base = p1 + unit * ti * dist
        offset = perp * amp * np.sin(freq * ti * dist)
        out[i] = base + offset

    return out

def get_zigzag_line(p1, p2, n=200, amp=0.1, freq=12, t_max=1.0):
    p1, p2 = np.array(p1), np.array(p2)
    dist = np.linalg.norm(p2 - p1)
    if dist == 0:
        return np.zeros((n, 2))

    t = np.linspace(0, t_max, n)
    unit = (p2 - p1) / dist
    perp = np.array([-unit[1], unit[0]])

    out = np.zeros((n, 2))

    for i, ti in enumerate(t):
        base = p1 + unit * ti * dist
        wave = np.sign(np.sin(freq * ti * dist))
        out[i] = base + perp * amp * wave

    return out

# ============================================================
# FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=(12.8, 7.2), dpi=100)
fig.patch.set_facecolor(BG_COLOR)

fig.text(
    0.98, 0.02,
    "Mugambi Ndwiga | @craftsandengineering",
    color=TEXT_COLOR,
    alpha=0.25,
    fontsize=9,
    ha='right'
)

# ============================================================
# CORE LOOP
# ============================================================

def init():
    ax.clear()
    ax.set_xlim(-4.5, 4.5)
    ax.set_ylim(-3.5, 3.5)
    ax.axis('off')
    return []

def update(frame):

    ax.clear()
    ax.set_xlim(-4.5, 4.5)
    ax.set_ylim(-3.5, 3.5)
    ax.axis('off')

    # Intro
    if frame < 30:
        draw_title(frame / 30)
        return []

    # Outro
    if frame > TOTAL_FRAMES - 30:
        draw_end()
        return []

    # Scenes
    tframe = frame - 30
    scene_len = 9 * FPS
    scene = int(tframe // scene_len)
    t = (tframe % scene_len) / scene_len

    scenes = [
        scene_1,
        scene_2,
        scene_3,
        scene_4,
        scene_5,
        scene_6
    ]

    if scene < len(scenes):
        scenes[scene](t)

    return []

# ============================================================
# TITLE / END
# ============================================================

def draw_title(t):
    a = ease_in_out(t)
    ax.text(0, 0.7, "The Higgs Mechanism", color=TEXT_COLOR, fontsize=26, ha='center', alpha=a)
    ax.text(0, 0.1, "Mass, Symmetry Breaking, and Particle Physics", color=TEXT_COLOR, fontsize=14, ha='center', alpha=a)

def draw_end():
    ax.text(0, 0.2, "Created by Mugambi Ndwiga", color=TEXT_COLOR, fontsize=20, ha='center')
    ax.text(0, -0.5, "@craftsandengineering", color=HIGGS_COLOR, fontsize=12, ha='center')

# ============================================================
# SCENE 1 — HIGGS FIELD
# ============================================================

def scene_1(t):
    ax.text(0, 3, "The Higgs Field", color=TEXT_COLOR, ha='center', fontsize=18)
    ax.text(0, -3.2, "A universal scalar field that generates inertial mass via quantum drag.", color=SUB_TEXT_COLOR, ha='center', fontsize=12)

    for x in np.linspace(-4, 4, 18):
        for y in np.linspace(-3, 3, 12):
            ax.plot(x, y, '.', color=HIGGS_COLOR, alpha=0.05)

    x = lerp(-3.5, 3.5, ease_in_out(t))
    ax.plot([-3.5, x], [0, 0], color=FERMION_COLOR)

    if x > 0:
        ax.plot([0, x], [0, 0], color=FERMION_COLOR, lw=3)
        ax.plot([0, 0], [0, 1.5 * fade(t, 0.4, 0.7)], '--', color=HIGGS_COLOR)

# ============================================================
# SCENE 2 — SYMMETRY BREAKING
# ============================================================

def scene_2(t):
    ax.text(0, 3, "Spontaneous Symmetry Breaking", color=TEXT_COLOR, ha='center', fontsize=18)
    ax.text(0, -3.2, "The vacuum settles into a non-zero state, breaking the electroweak symmetry.", color=SUB_TEXT_COLOR, ha='center', fontsize=12)

    theta = np.linspace(0, 2*np.pi, 300)
    r = 1.2
    ax.plot(r*np.cos(theta), 0.6*r*np.sin(theta)-0.5, color=HIGGS_COLOR)

    ax.plot(
        lerp(0, 1.2, ease_in_out(t)),
        lerp(0.5, -0.5, ease_in_out(t)),
        'o',
        color=FLASH_COLOR
    )

# ============================================================
# SCENE 3 — GLUON FUSION
# ============================================================

def scene_3(t):
    ax.text(0, 3, "Gluon Fusion via Top Loop", color=TEXT_COLOR, ha='center', fontsize=18)
    ax.text(0, -3.2, "Colliding gluons couple to heavy top quarks to excite the Higgs field into a particle.", color=SUB_TEXT_COLOR, ha='center', fontsize=12)

    g = get_zigzag_line([-3.5, 1], [-1.2, 0.5], t_max=fade(t, 0, 0.4))
    ax.plot(g[:,0], g[:,1], color=GLUON_COLOR)

    g = get_zigzag_line([-3.5, -1], [-1.2, -0.5], t_max=fade(t, 0, 0.4))
    ax.plot(g[:,0], g[:,1], color=GLUON_COLOR)

    lt = fade(t, 0.3, 0.7)
    loop = np.array([[-1.2, 0.5], [0.2, 1.0], [0.2, -1.0], [-1.2, 0.5]])

    for i in range(3):
        p0, p1 = loop[i], loop[i+1]
        ax.plot([p0[0], lerp(p0[0], p1[0], lt)], [p0[1], lerp(p0[1], p1[1], lt)], color=TOP_COLOR)

    if t > 0.7:
        ax.plot([0.2, 2.2], [0, 0.8], '--', color=HIGGS_COLOR)

# ============================================================
# SCENE 4 — H → γγ
# ============================================================

def scene_4(t):
    ax.text(0, 3, r"$H \rightarrow \gamma\gamma$", color=TEXT_COLOR, ha='center', fontsize=18)
    ax.text(0, -3.2, "The Higgs boson decays via quantum loops into two characteristic high-energy photons.", color=SUB_TEXT_COLOR, ha='center', fontsize=12)

    ax.plot([-3.5, -1.5], [0, 0], '--', color=HIGGS_COLOR)
    wt = fade(t, 0.3, 0.7)
    theta = np.linspace(0, 2*np.pi*wt, 200)
    ax.plot(np.cos(theta), np.sin(theta), color=W_COLOR)

    pt = fade(t, 0.6, 0.95)
    if pt > 0:
        p = get_wavy_line([0,0.6],[3,1.8], t_max=pt)
        ax.plot(p[:,0], p[:,1], color=PHOTON_COLOR)

# ============================================================
# SCENE 5 — GOLDEN CHANNEL
# ============================================================

def scene_5(t):
    ax.text(0, 3, r"Golden Channel: H → ZZ* → 4ℓ", color=TEXT_COLOR, ha='center', fontsize=18)
    ax.text(0, -3.2, "A rare and precise decay mode producing four leptons with a distinct mass signature.", color=SUB_TEXT_COLOR, ha='center', fontsize=12)

    ax.plot([-4,-2], [0,0], '--', color=HIGGS_COLOR)
    zt = fade(t, 0.2, 0.5)
    if zt > 0:
        ax.plot([-2,-0.5], [0,1], color=Z_COLOR)
        ax.plot([-2,-0.5], [0,-1], color=Z_COLOR)

# ============================================================
# SCENE 6 — EW BREAKING
# ============================================================

def scene_6(t):
    ax.text(0, 3, "Electroweak Symmetry Breaking", color=TEXT_COLOR, ha='center', fontsize=18)
    ax.text(0, -3.2, "The Higgs field differentiates the electroweak force into short-range and long-range forces.", color=SUB_TEXT_COLOR, ha='center', fontsize=12)

    x = lerp(-1, 4, t)
    ax.axvline(x, color=HIGGS_COLOR, alpha=0.3)

# ============================================================
# ANIMATION
# ============================================================

ani = FuncAnimation(fig, update, frames=TOTAL_FRAMES, init_func=init, blit=False)

# ============================================================
# EXPORT
# ============================================================

writer = FFMpegWriter(fps=FPS, bitrate=2600)
out = "higgs_final.mp4"
ani.save(out, writer=writer)

# ============================================================
# DISPLAY
# ============================================================

mp4 = open(out, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

display(HTML(f"""
<video width=960 controls autoplay loop>
<source src=\"{data_url}\" type=\"video/mp4\">
</video>
"""))

google.colab.files.download(out)
plt.close(fig)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML
from base64 import b64encode

# --- CONFIG ---
BG_COLOR = '#050510'
FPS = 30
SCENE_FRAMES = 150
TOTAL_FRAMES = SCENE_FRAMES * 2

fig = plt.figure(figsize=(12.8, 7.2), facecolor=BG_COLOR)
ax = fig.add_subplot(111, projection='3d')
ax.set_facecolor(BG_COLOR)

# Precompute Mexican Hat
r_hat = np.linspace(0, 1.5, 40)
theta_hat = np.linspace(0, 2*np.pi, 40)
R, TH = np.meshgrid(r_hat, theta_hat)
X_H = R * np.cos(TH)
Y_H = R * np.sin(TH)
Z_H = (R**2 - 1)**2

def update_combined(frame):
    ax.clear()
    ax.axis('off')

    if frame < SCENE_FRAMES:
        # --- SCENE 1: 3D POTENTIAL ---
        t = frame / SCENE_FRAMES
        ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_zlim(0, 1.5)
        ax.plot_surface(X_H, Y_H, Z_H, cmap='YlOrBr', alpha=0.2, antialiased=True)

        # Rolling particle
        cur_r = 1.0 * (1 - np.exp(-4*t))
        cur_th = t * 3
        px, py = cur_r * np.cos(cur_th), cur_r * np.sin(cur_th)
        pz = (cur_r**2 - 1)**2
        ax.plot([px], [py], [pz], 'o', color='#ffffff', markersize=10)

        ax.text2D(0.5, 0.9, "Spontaneous Symmetry Breaking", transform=ax.transAxes, color='#f5f5f5', size=16, ha='center')
        if t > 0.7: ax.text2D(0.5, 0.8, "Vacuum Expectation Value ≠ 0", transform=ax.transAxes, color='#ffb300', size=12, ha='center')
        ax.view_init(elev=30, azim=frame)

    else:
        # --- SCENE 2: 3D GLUON FUSION ---
        t = (frame - SCENE_FRAMES) / SCENE_FRAMES
        ax.set_xlim(-2, 2); ax.set_ylim(-2, 2); ax.set_zlim(-2, 2)

        # Incoming Gluons (3D ZigZags)
        for side in [1, -1]:
            pts = np.linspace(0, np.clip(t*2, 0, 1), 50)
            gx = -2 + pts * 1.5
            gy = side * (1 - pts)
            gz = side * (1 - pts) + 0.1 * np.sin(pts * 20)
            ax.plot(gx, gy, gz, color='#76ff03', lw=1.5)

        # Triangle Top Loop
        if t > 0.5:
            lt = np.clip((t-0.5)*2, 0, 1)
            lpts = np.array([[-0.5,0,0], [0,0.5,0.5], [0,-0.5,0.5], [-0.5,0,0]])
            for i in range(3):
                st = np.clip(lt*3-i, 0, 1)
                p1, p2 = lpts[i], lpts[i+1]
                curr = p1 + (p2-p1)*st
                ax.plot([p1[0], curr[0]], [p1[1], curr[1]], [p1[2], curr[2]], color='#ff7043', lw=2)

        if t > 0.8:
            ax.plot([0, 1.5], [0,0], [0.5, 0.5], '--', color='#ffb300', lw=2)
            ax.text(1.6, 0, 0.5, "Higgs (H)", color='#ffb300')

        ax.text2D(0.5, 0.9, "3D Gluon Fusion (LHC Production)", transform=ax.transAxes, color='#f5f5f5', size=16, ha='center')
        ax.view_init(elev=20, azim=frame*0.5)

    return []

ani_3d_film = FuncAnimation(fig, update_combined, frames=TOTAL_FRAMES, interval=40)
output_final = 'higgs_3d_film.mp4'
ani_3d_film.save(output_final, writer=FFMpegWriter(fps=30))

mp4_3d = open(output_final,'rb').read()
display(HTML(f'<video width=960 controls autoplay loop><source src="data:video/mp4;base64,{b64encode(mp4_3d).decode()}" type="video/mp4"></video>'))
plt.close(fig)